# Proyecto Gestión de Reservas Horizon Austral

# 1.1. Crear estructura Base de Datos

In [1]:
import sqlite3
import pandas as pd

with sqlite3.connect('horizon_austral_operaciones.db') as conn:
    cursor=conn.cursor()
    cursor.execute('''
    DROP TABLE IF EXISTS Hechos_Reservas''')
    cursor.execute('''
    DROP TABLE IF EXISTS Dim_Clientes''')
    cursor.execute('''
    DROP TABLE IF EXISTS Dim_Tours''')

In [2]:
import sqlite3 
import pandas as pd

with sqlite3.connect('horizon_austral_operaciones.db') as conn:
    cursor=conn.cursor()
#crea tabla tours
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Dim_Tours (
        id_tour INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_tour TEXT NOT NULL,
        capacidad_maxima INTEGER,
        precio_base REAL)
        ''')

#crea tabla Clientes
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Dim_Clientes (
        dni_pasaporte TEXT PRIMARY KEY,
        nombre_completo TEXT NOT NULL,
        edad INTEGER,
        pais TEXT,
        email TEXT) 
        ''')
#Crea tabla de hechos, reservas
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Hechos_Reservas (
        id_reserva INTEGER PRIMARY KEY AUTOINCREMENT,
        fecha_reserva DATE NOT NULL,
        dni_pasaporte TEXT,
        id_tour INTEGER,
        monto_total REAL,
        FOREIGN KEY (dni_pasaporte) REFERENCES Dim_Clientes (dni_pasaporte),
        FOREIGN KEY (id_tour) REFERENCES Dim_Tours (id_tour))
        ''')
    print("Estructura profesional de Horizon Austral creada exitosamente!")
    
    

Estructura profesional de Horizon Austral creada exitosamente!


## 1.2. Inserción Datos Maestros 

In [3]:
def poblar_datos_iniciales():
    with sqlite3.connect('horizon_austral_operaciones.db') as conn:
        cursor=conn.cursor()
#insertar tours iniciales
        tours = [('Grande Decouverte',8,2500),
                 ('Gauchos',9,1800),
                 ('Cordillera Darwin',8,2000),
                 ('Route Australe',8,1700),
                 ('Personalizada',5,2300)]
        tours_insertados=len(tours)
        cursor.executemany('''
            INSERT INTO Dim_Tours (nombre_tour, capacidad_maxima,precio_base)
            VALUES (?,?,?) ''', tours)
#insertar clientes
        clientes=[('20444555k','Nicolas Melphi',28,'Argentina','nicomelphi@example.com'),
                  ('PA9988777','Elena Smith',36,'USA','elena.s@example.com'),
                  ('181234567','Francisco Pérez',42,'Chile','f_perez@example.com')]
        clientes_insertados=len(clientes)
     
        cursor.executemany('''
            INSERT INTO Dim_Clientes (dni_pasaporte,nombre_completo,edad,pais,email)
            VALUES (?,?,?,?,?)''',clientes)

        print(f"Se han insertado {tours_insertados} tours y {clientes_insertados} clientes")
        cursor.execute('SELECT count(*) FROM Dim_Tours')
        total_tours=cursor.fetchone()[0]
        print(f" total de tours en la base de datos: {total_tours}")
poblar_datos_iniciales()

Se han insertado 5 tours y 3 clientes
 total de tours en la base de datos: 5


## 1.3. Creamos 200 clientes más

In [9]:
import random
import sqlite3

def generar_clientes(n_clientes=200):
    with sqlite3.connect('horizon_austral_operaciones.db') as conn:
        cursor=conn.cursor()

    paises=['Francia','Belgica','Suiza','Canada','Japon','Reino Unido','España','Alemania']
    nuevos_clientes=[]
    for i in range(200):
        dni_pasaporte=f"CLI-{1000+i}"
        nombre_completo=f"Cliente {i}"
        edad=random.randint(40,80)
        pais = random.choice(paises)
        email=f"Cliente{i}+@example.com"
        nuevos_clientes.append((dni_pasaporte,nombre_completo,edad,pais,email))

    cursor.executemany(''' INSERT INTO Dim_Clientes (dni_pasaporte,nombre_completo,edad,pais,email) 
                VALUES (?,?,?,?,?)''', nuevos_clientes)
    conn.commit()
    print(f" Se han insertado correctamente {n_clientes} nuevos clientes")

generar_clientes(200)

 Se han insertado correctamente 200 nuevos clientes


## 1.4. Simulamos una gran cantidad de reservas aleatorias

In [10]:
import random
from datetime import datetime, timedelta

def generar_reservas_masivas(n_reservas=500):
    with sqlite3.connect('horizon_austral_operaciones.db') as conn:
        cursor=conn.cursor()
        cursor.execute('''
            SELECT dni_pasaporte FROM Dim_Clientes''')
        listado_clientes=[fila[0] for fila in cursor.fetchall()]


        
        cursor.execute('''SELECT id_tour,precio_base FROM Dim_Tours''')
        listado_tours=cursor.fetchall()

        reservas_para_insertar=[]
        fecha_inicio=datetime(2026,4,12)
        for _ in range(n_reservas):
            cliente=random.choice(listado_clientes)
            tour_id,precio_base=random.choice(listado_tours)
# generamos una fecha aleatoria en los últimos meses
            dias_aleatorios=random.randint(0,180)
            fecha_reserva=(fecha_inicio + timedelta(days=dias_aleatorios)).date()

#lógica de negocios, descuentos aleatorios
            descuento=random.choice([1,0.95,0.9])
            monto_final=round(precio_base * descuento,2)

            reservas_para_insertar.append((fecha_reserva.isoformat(),cliente,tour_id,monto_final))

#insersión masiva
        cursor.executemany('''
        INSERT INTO Hechos_Reservas (fecha_reserva, dni_pasaporte,id_tour,monto_total)
        VALUES (?,?,?,?)''',reservas_para_insertar)
        conn.commit()
        print(f" ¡proceso completado! Se han generado {n_reservas} reservas aleatorias.")

generar_reservas_masivas(500)
        
        

 ¡proceso completado! Se han generado 500 reservas aleatorias.


In [11]:
with sqlite3.connect('horizon_austral_operaciones.db') as conn:    
    resultado=pd.read_sql_query('SELECT * FROM Hechos_Reservas', conn)
    print(resultado.head(10))

   id_reserva fecha_reserva dni_pasaporte  id_tour  monto_total
0           1    2026-09-19      CLI-1183        2       1710.0
1           2    2026-10-07      CLI-1148        3       1800.0
2           3    2026-09-30      CLI-1121        3       1900.0
3           4    2026-09-09      CLI-1091        2       1710.0
4           5    2026-08-21      CLI-1074        2       1710.0
5           6    2026-04-25      CLI-1000        2       1710.0
6           7    2026-05-15      CLI-1141        1       2375.0
7           8    2026-06-22      CLI-1145        2       1620.0
8           9    2026-05-08      CLI-1080        1       2250.0
9          10    2026-05-01      CLI-1173        5       2070.0


## 1.5. Exportar dataframes a formato CSV para trabajar en Power BI

In [12]:
with sqlite3.connect('horizon_austral_operaciones.db') as conn:
    # Exportamos las 3 tablas a CSV
    pd.read_sql_query("SELECT * FROM Hechos_Reservas", conn).to_csv('hechos_reservas.csv', index=False)
    pd.read_sql_query("SELECT * FROM Dim_Clientes", conn).to_csv('dim_clientes.csv', index=False)
    pd.read_sql_query("SELECT * FROM Dim_Tours", conn).to_csv('dim_tours.csv', index=False)
    
print("¡Listo! Ya tienes los 3 archivos .csv en tu carpeta. Ábrelos en Power BI.")

¡Listo! Ya tienes los 3 archivos .csv en tu carpeta. Ábrelos en Power BI.
